### Plotting enrichments of parent protein functional categories by peptide hit status 

General approach:

- Take the GO slimming sets generated in processing file (Figure_S2E_Processing.ipynb) 
- text mine these to sort into 5 categories (curated set and mapping)
- Fischer's test to test for enrichment

In [ ]:
#import annotations csv 
#!
import pandas as pd 
import matplotlib.pyplot as plt
annotations_df = pd.read_csv("GO_Term_Mapped_Viral_Proteins.csv", header=0)
#filter on non go term having entires 

annotations_df = annotations_df[~(annotations_df['GO IDs']=='[]')]
print(annotations_df.head().to_string())
print(f"Total number of hits: {len(annotations_df)}")

In [ ]:
#now we need to see how many have any GO terms...
annotations_df['Amt_GO'] = annotations_df['GO IDs'].apply(lambda x: len(x.split(' ')))
print(annotations_df.head().to_string())


In [ ]:
#make sure we listify 
import ast

GOTERM_COL = []
for index, row in annotations_df.iterrows():
    Goterm_string = row['GO Terms']
    Goterm_list = ast.literal_eval(Goterm_string)
    GOTERM_COL.append(Goterm_list)
print(GOTERM_COL)
annotations_df['GO Terms'] = GOTERM_COL

In [ ]:

Structure_Assembly = ["envelope", "capsid", "structural", "structure" "procapsid", "assembly", "coating", "uncoating", "virion membrane", "genome packaging"]
Modulation =  ["immune", "suppression", "perturbation", "symbiont", 'response to type I interferon', 'symbiotic']
Replication_Organization = ["genome replication", "integration","DNA binding", "RNA binding", "DNA recombination", "RNA polymerase"]
Replication_Machinery = ["mRNA 5'-cap (guanine-N7-)-methyltransferase activity","DNA helicase", "ribonucleoprotein", "DNA polymerase", "helicase", "viral transcription"]
Entry_Exit = ["attatchment", "entry", "exit", "fusion", "budding", "CD4", "release", "host cell surface"]


def Categorize_Golist(Golist):
    Structure_Assembly = ["envelope", "capsid", "structural", "structure" "procapsid", "assembly", "coating", "uncoating", "virion membrane", "genome packaging"]
    Modulation =  ["immune", "suppression", "perturbation", "symbiont", "protein serine/threonine kinase inhibitor activity"]
    Replication_Organization = ["genome replication", "integration","DNA binding", "RNA binding", "DNA recombination"]
    Replication_Machinery = ["mRNA 5'-cap (guanine-N7-)-methyltransferase activity","DNA helicase", "ribonucleoprotein", "DNA polymerase", "helicase", "viral transcription"]
    Entry_Exit = ["attatchment", "entry", "exit", "fusion", "budding", "CD4", "release", "host cell surface"]

    """ 
    IN: Golist (list of go terms)
    OUT: 5-tuple where first 5 terms correspond to booleans for 
    categorization into the first 5 categories. 
    """
    Struct_Assembly_Present = False
    Modulation_Present = False 
    Replication_Organization_Present = False
    Replication_Machinery_Present =  False 
    Entry_Exit_Present = False 
    
    if any(term in x for x in Golist for term in Structure_Assembly):
        Struct_Assembly_Present = True

    if any(term in x for x in Golist for term in Modulation):
        Modulation_Present = True

    if any(term in x for x in Golist for term in Replication_Organization):
        Replication_Organization_Present = True

    if any(term in x for x in Golist for term in Replication_Machinery):
        Replication_Machinery_Present = True

    if any(term in x for x in Golist for term in Entry_Exit):
        Entry_Exit_Present = True
    
    
    return pd.Series({'Structure_Assembly':Struct_Assembly_Present, 'Modulation':Modulation_Present, 'Replication_Organization':Replication_Organization_Present, 'Replication_Machinery':Replication_Machinery_Present, 'Entry_Exit':Entry_Exit_Present})

In [ ]:
#add annotations
annotations_df[['Structure_Assembly','Modulation','Replication_Organization','Replication_Machinery','Entry_Exit']] = annotations_df['GO Terms'].apply(lambda x: Categorize_Golist(x))


In [ ]:
#collapse on shared NCBI ID if ncbi ID exits

collapse_df = pd.DataFrame(columns=['Protein ID', 'Reviewed Used', 'Structure_Assembly','Modulation','Replication_Organization','Replication_Machinery','Entry_Exit'])

def trueif(list):
    return True in list

grouped = annotations_df.groupby('NCBI ID')



for name, group in grouped:
    if True in group['Reviewed'].to_list():
        group_rev = group[group['Reviewed']==True]
        
        collapse_df.loc[len(collapse_df)] = [group['NCBI ID'].unique().tolist()[0], True, trueif(group_rev['Structure_Assembly'].to_list()), trueif(group_rev['Modulation'].to_list()), trueif(group_rev['Replication_Organization'].to_list()), trueif(group_rev['Replication_Machinery'].to_list()), trueif(group_rev['Entry_Exit'].to_list())]
    else:
        collapse_df.loc[len(collapse_df)] = [group['NCBI ID'].unique().tolist()[0], False, trueif(group['Structure_Assembly'].to_list()), trueif(group['Modulation'].to_list()), trueif(group['Replication_Organization'].to_list()), trueif(group['Replication_Machinery'].to_list()), trueif(group['Entry_Exit'].to_list())]



#now for the others...

annotations_df_onlyoriginalUPID = annotations_df[(annotations_df['NCBI ID'].isnull())]

grouped = annotations_df_onlyoriginalUPID.groupby(['Uniprot ID'])
for name, group in grouped:

    if True in group['Reviewed'].to_list():
        group_rev = group[group['Reviewed']==True]
        collapse_df.loc[len(collapse_df)] = [group['Uniprot ID'].unique().tolist()[0], True, trueif(group_rev['Structure_Assembly'].to_list()), trueif(group_rev['Modulation'].to_list()), trueif(group_rev['Replication_Organization'].to_list()), trueif(group_rev['Replication_Machinery'].to_list()), trueif(group_rev['Entry_Exit'].to_list())]
    else:
        collapse_df.loc[len(collapse_df)] = [group['Uniprot ID'].unique().tolist()[0], False, trueif(group['Structure_Assembly'].to_list()), trueif(group['Modulation'].to_list()), trueif(group['Replication_Organization'].to_list()), trueif(group['Replication_Machinery'].to_list()), trueif(group['Entry_Exit'].to_list())]



In [ ]:
#now we get these for the subsets
#bring in other data


#clean it up!!!


#loaad all data
df_rita_pos_selection = pd.read_excel(r"RITA_and_ABT_pos_selection_screens.xlsx",sheet_name = "RITA")
#filter poly
df_rita_pos_selection = df_rita_pos_selection[~df_rita_pos_selection['tileID'].str.contains('H')]
#filter poly
df_rita_pos_selection = df_rita_pos_selection[~df_rita_pos_selection['Gene_name'].str.contains('polyprotein')]
#filter non human positive 
df_rita_pos_selection = df_rita_pos_selection[(~df_rita_pos_selection['tileID'].str.contains('P')) | (~df_rita_pos_selection['tileID'].str.contains('N'))]


def test_effect(row):
    
    if row['sig'] == 'No':
        return 'None' 
    elif row['log2FoldChange'] > 0:
        return 'Protective'
    else:
        return 'Deleterious'
    
df_rita_pos_selection['Effect'] = df_rita_pos_selection.apply(lambda row: test_effect(row), axis=1)


#map ID to effect
map_ID_effect_protein = {}


for protein_id in collapse_df['Protein ID']:
    # 1. Check both ID columns for a match
    mask_uniprot = df_rita_pos_selection['Uniprot_id'] == protein_id
    mask_ncbi = df_rita_pos_selection['NCBI_id'] == protein_id
    
    # Combine masks to find any row where either ID matches
    filtered = df_rita_pos_selection[mask_uniprot | mask_ncbi]

    if not filtered.empty:
        # Get unique effects found for this specific protein
        found_effects = filtered['Effect'].unique()
        
        has_deleterious = 'Deleterious' in found_effects
        has_protective = 'Protective' in found_effects

        if has_deleterious and has_protective:
            map_ID_effect_protein[protein_id] = 'Mixed'
        elif has_deleterious:
            map_ID_effect_protein[protein_id] = 'Deleterious'
        elif has_protective:
            map_ID_effect_protein[protein_id] = 'Protective'
        else:
            map_ID_effect_protein[protein_id] = 'None'
    else:
        # Optional: handle cases where the protein_id isn't found at all
        map_ID_effect_protein[protein_id] = 'Not Found'
      

import pandas as pd

def annotate_function_for_row(row):
    cols = ['Structure_Assembly', 'Modulation', 'Replication_Organization', 'Replication_Machinery', 'Entry_Exit']
    
    # Check Uniprot_id
    mask_uniprot = collapse_df['Protein ID'] == row["Uniprot_id"]
    # Check NCBI_id
    mask_ncbi = collapse_df['Protein ID'] == row['NCBI_id']
    
    if mask_uniprot.any():
        # .iloc[0] ensures we get a single row as a Series
        res = collapse_df.loc[mask_uniprot, cols].iloc[0]
    elif mask_ncbi.any():
        res = collapse_df.loc[mask_ncbi, cols].iloc[0]
    else:
        # Return a Series of NAs with the correct index
        res = pd.Series([pd.NA] * len(cols), index=cols)
    
    return res

# Use result_type='expand' to turn the returned Series into separate columns
target_cols = ['Structure_Assembly', 'Modulation', 'Replication_Organization', 'Replication_Machinery', 'Entry_Exit']
df_rita_pos_selection[target_cols] = df_rita_pos_selection.apply(annotate_function_for_row, axis=1, result_type='expand')
df_rita_pos_selection['Replication_General'] = df_rita_pos_selection['Replication_Organization'] | df_rita_pos_selection['Replication_Machinery']




In [ ]:
df_rita_pos_selection_onlyfunctions = df_rita_pos_selection.dropna(subset=['Structure_Assembly'])

df_rita_pos_selection_onlyfunctions_noeffect = df_rita_pos_selection_onlyfunctions[df_rita_pos_selection_onlyfunctions['Effect']=='None']
df_rita_pos_selection_onlyfunctions_protective = df_rita_pos_selection_onlyfunctions[df_rita_pos_selection_onlyfunctions['Effect']=='Protective']
df_rita_pos_selection_onlyfunctions_deleterious = df_rita_pos_selection_onlyfunctions[df_rita_pos_selection_onlyfunctions['Effect']=='Deleterious']




In [ ]:
#now perform fischer's

import scipy
from scipy import stats

fischer_data = pd.DataFrame(columns=['Subset', 'Functional Category', 'Pvalue', 'Odds Ratio'])

""" 

tables...

               | Other | Background |
Not Modulation |     a         b

Modulation     |     c          d


"""
from scipy.stats import fisher_exact
from scipy.stats.contingency import odds_ratio




for effect in ['None', 'Deleterious', 'Protective']:

    
    for category in ['Structure_Assembly','Modulation','Replication_Organization','Replication_Machinery','Entry_Exit', 'Replication_General']:
        row = []
        subset_df = df_rita_pos_selection_onlyfunctions[df_rita_pos_selection_onlyfunctions['Effect']==effect]
        
        # Background counts
        Background_Category = df_rita_pos_selection_onlyfunctions[category].sum()  # Count of True values
        Background_Not_Category = len(df_rita_pos_selection_onlyfunctions) - Background_Category
        
        # Effect subset counts
        Effect_Category = subset_df[category].sum()  # Count of True values
        Effect_Not_Category = len(subset_df) - Effect_Category
        
        # Contingency table: [[Not_Category_Background, Not_Category_Effect], [Category_Background, Category_Effect]]
        
        #print(f"\t\t\t| Background | {effect} |\nNot {category} | {Background_Not_Category} | {Background_Category}\n\t{category} | {Background_Category} | {Effect_Category}")
        
        table = [[Effect_Category, Background_Category], 
                 [Effect_Not_Category, Background_Not_Category]]
        
        print(f"Testing the proportion of {category} in tiles with {effect} effect\n\n")
        temp_table_df = pd.DataFrame(columns=[ effect, 'Background'], index=[category, f"Not {category}"])
        temp_table_df.loc[category] = [ Effect_Category,Background_Category]
        temp_table_df.loc[f"Not {category}"] = [Effect_Not_Category, Background_Not_Category]
        print("Following is the contingency table, with null hypothesis that background has the same\nproportion of viral tiles with this category under the hypergeometric distribution\n\n")
        print(temp_table_df)
        print("\n\n")
        
        res = fisher_exact(table)

        print(f"Probability of observing a contingency table at least this extreme under the null distribution: {res.pvalue*100:.2f}%")
        odds_background = Background_Category / Background_Not_Category
        odds_subset = Effect_Category / Effect_Not_Category
        print(f"(Pvalue: {res.pvalue:.3f})\n\n")
        print(f"Odds of observing {category} in background: {odds_background}\nOdds of observing {category} in peptides with {effect} effect: {odds_subset}")
        
        
        odds_ratio = odds_subset / odds_background
        
        print(f"Odds ratio: {odds_ratio}\n")
        
        if res.pvalue < 0.05: 
            if odds_ratio < 1:
                
                print(f"Significant depletion of {category} in peptides with {effect} effect")
        
            else:
                
                print(f"Significant enrichment of {category} in peptides with {effect} effect")

            
            
        print(f"------------------------------------------------------------------\n\n\n")


  
        mapping ={'Structure_Assembly': "Structure & Assembly",'Modulation': "Modulation",'Replication_Organization': "Organization of Replication",'Replication_Machinery': "Replication Machinery",'Entry_Exit': "Viral Entry & Exit", "Replication_General": "Rep Gen"}

        row=[effect, mapping[category], res.pvalue, odds_ratio]

        fischer_data.loc[len(fischer_data)] = row
        

In [ ]:
#plotting

from matplotlib import font_manager

from matplotlib import rcParams
rcParams['font.sans-serif'] = "Arial"
rcParams['font.family'] = "Arial"



plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial'

import seaborn as sns 
import numpy as np
import math
custom_palette=sns.color_palette(["#A5A8A8","#36B1AD","#ECCC72"])

fig, ax = plt.subplots(figsize=(6,4))
print(fischer_data)
mapping = {"Subset": "Peptide Effect", "Functional Category": "Viral Parent Protein Function"}

fischer_data = fischer_data.rename(columns = mapping)
fischer_data['Pvalue'] = fischer_data['Pvalue'].apply(lambda x: -1*math.log10(x))
print(fischer_data)
ax = sns.scatterplot(data=fischer_data, x='Odds Ratio', y='Pvalue', hue="Peptide Effect", palette=custom_palette, style='Viral Parent Protein Function', s=100)
#plt.legend(title="")

import matplotlib.pyplot as plt
import matplotlib.text as mtext

ax.spines['left'].set_linewidth(1.5)
ax.yaxis.set_tick_params(width=1.5)
plt.xlim(0.4, 1.6)
plt.ylim(0,5)
ax.xaxis.set_tick_params(width=1.5)
ax.spines['bottom'].set_linewidth(1.5)

sns.move_legend(ax, "upper left", bbox_to_anchor=(1,1))
plt.axhline(y=-1*math.log10(0.05), color='black', linestyle='solid', lw=1.0)
plt.ylabel("-log(P-value)")
sns.despine(offset=10, trim=True)
plt.tight_layout()
plt.savefig("Functional_Categories_S2E.pdf", dpi=300)